In [1]:
!pip install --quiet deepchem rdkit transformers accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 11.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 58.0 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install --quiet trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 11.0 MB/s eta 0:00:00a 0:00:01


In [5]:
import os, gc, torch, numpy as np, pandas as pd
import deepchem as dc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# memory cleanup
torch.cuda.empty_cache()
gc.collect()

# --- NEW STRATEGY: DIRECT DATA HANDLING (BYPASSING DEEPCHEM LOADER CRASH) ---
# Instead of relying on the buggy loader, we fetch the CSV and clean it manually.
print("Downloading and Cleaning BBBP Data...")
data_url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(data_url)

# Clean bad SMILES (The cause of your error)
# We check if RDKit can parse them. If not, drop them.
from rdkit import Chem
valid_indices = []
for idx, row in df.iterrows():
    mol = Chem.MolFromSmiles(row['smiles'])
    if mol is not None:
        valid_indices.append(idx)

df_clean = df.iloc[valid_indices].reset_index(drop=True)
print(f"Original: {len(df)} -> Cleaned: {len(df_clean)}")

# --- STRATEGIC SCAFFOLD SPLIT ---
# We use DeepChem ONLY for the split logic to keep benchmarks valid
print("Applying Scaffold Split...")
dataset = dc.data.NumpyDataset(X=df_clean['smiles'].values, y=df_clean['p_np'].values, ids=df_clean['smiles'].values)
splitter = dc.splits.ScaffoldSplitter()
train_dc, valid_dc, test_dc = splitter.train_valid_test_split(dataset)

print(f"Train: {len(train_dc)}, Test: {len(test_dc)}")

# Dataset Class
class BBBPDataset(Dataset):
    def __init__(self, dc_data, tokenizer, max_len=256, upsample=False):
        self.data = []
        self.tokenizer = tokenizer
        
        X = dc_data.X # SMILES
        y = dc_data.y # Labels
        
        indices = range(len(X))
        
        # Balancing Strategy (Mistral needs to see enough negatives)
        if upsample:
            y_flat = y.flatten()
            pos = np.where(y_flat==1)[0]
            neg = np.where(y_flat==0)[0]
            # 3x upsampling for negatives (minority)
            indices = list(pos) + list(neg) * 3
            np.random.shuffle(indices)
            
        for i in indices:
            smi = str(X[i])
            label = int(y[i])
            label_str = "Yes" if label == 1 else "No"
            
            # Prompt: Short and Scientific
            txt = f"Task: BBBP Permeability | SMILES: {smi} | Permeable: {label_str}" + tokenizer.eos_token
            
            enc = tokenizer(
                txt, 
                max_length=max_len, 
                padding="max_length", 
                truncation=True, 
                return_tensors="pt"
            )
            self.data.append({
                "ids": enc["input_ids"][0],
                "mask": enc["attention_mask"][0],
                "label": label
            })

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# Model Init
print("Loading Mistral-7B...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_id = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Dataloaders
train_ds = BBBPDataset(train_dc, tokenizer, upsample=True)
test_ds = BBBPDataset(test_dc, tokenizer, upsample=False)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)

opt = torch.optim.AdamW(model.parameters(), lr=2e-4)

# Training Config
EPOCHS = 5  # 5 is the sweet spot for BBBP on LLMs
ACCUM_STEPS = 8

print(f"Starting training loop for {EPOCHS} epochs...")

best_auc = 0.0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Ep {epoch+1}")
    
    for step, batch in enumerate(pbar):
        ids = batch["ids"].to("cuda")
        mask = batch["mask"].to("cuda")
        
        out = model(input_ids=ids, attention_mask=mask, labels=ids)
        loss = out.loss / ACCUM_STEPS
        loss.backward()
        
        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            opt.step()
            opt.zero_grad()
            
        epoch_loss += loss.item() * ACCUM_STEPS
        pbar.set_postfix({"loss": f"{epoch_loss/(step+1):.4f}"})

    # Validation
    print("Validating...")
    model.eval()
    preds, acts = [], []
    id_yes = tokenizer.encode("Yes", add_special_tokens=False)[0]
    id_no = tokenizer.encode("No", add_special_tokens=False)[0]
    
    with torch.no_grad():
        for item in tqdm(test_ds.data, desc="Testing"):
            full_txt = tokenizer.decode(item["ids"], skip_special_tokens=True)
            # Cut off the answer for inference
            query = full_txt.split("Permeable:")[0] + "Permeable:"
            
            inp = tokenizer(query, return_tensors="pt").to("cuda")
            out = model(**inp)
            logits = out.logits[0, -1, [id_no, id_yes]]
            probs = torch.nn.functional.softmax(logits.float(), dim=-1)
            
            preds.append(probs[1].item())
            acts.append(item["label"])
            
    auc = roc_auc_score(acts, preds)
    print(f"Epoch {epoch+1} BBBP ROC-AUC: {auc:.4f}")
    
    # Checkpoint logic: Save if we are beating or close to SOTA (0.724)
    if auc > best_auc:
        best_auc = auc
        if auc > 0.70:
            print(f"New Best Score ({auc:.4f})! Saving adapter...")
            model.save_pretrained(f"mistral_bbbp_best_epoch_{epoch+1}")

print(f"Final Best ROC-AUC: {best_auc:.4f}")

[13:56:57] Explicit valence for atom # 1 N, 4, is greater than permitted
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] WARNING: not removing hydrogen atom without neighbors
[13:56:57] Explicit valence for atom # 11 N, 4, is greater than pe

Original: 2050 -> Cleaned: 2039
Applying Scaffold Split...


[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not removing hydrogen atom without neighbors
[13:56:58] WARNING: not r

Train: 1631, Test: 204
Loading Mistral-7B...


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

...

Epoch 4 BBBP ROC-AUC: 0.6893


Ep 5:   0%|          | 0/1106 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Ep 5: 100%|██████████| 1106/1106 [34:44<00:00,  1.89s/it, loss=5.8118]


Validating...


Testing: 100%|██████████| 204/204 [00:44<00:00,  4.60it/s]

Epoch 5 BBBP ROC-AUC: 0.6720
Final Best ROC-AUC: 0.6904


In [6]:
# RUN 2: BBBP WITH CHEMICAL AUGMENTATION (Anti-Overfitting)
import torch
import numpy as np
import pandas as pd
from rdkit import Chem
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# 1. OPTIMIZER RESET (Purana learning rate bhool jao)
# Hum learning rate thora kam kar rahe hain taake model stable rahe
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # 2e-4 -> 1e-4

# 2. AUGMENTATION FUNCTION
def get_random_smiles(smiles):
    """Generates a random valid SMILES string for the same molecule."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        return Chem.MolToSmiles(mol, doRandom=True, canonical=False)
    except:
        return smiles

# 3. AUGMENTED DATASET CLASS
class AugmentedBBBPDataset(Dataset):
    def __init__(self, dc_data, tokenizer, max_len=256, augment=False):
        self.data = []
        self.tokenizer = tokenizer
        
        X = dc_data # These are already numpy arrays from our previous cell
        y = dc_data
        
        # Logic: Agar Augment=True hai (Training), toh hum har molecule ke 
        # 3 random versions banayenge.
        
        # BBBP Data (Cleaned DataFrame se uthayenge jo memory mein hai: df_clean)
        # HACK: Hum direct train_dc object use kar rahe hain jo pichle cell mein banaya tha
        ids = dc_data.ids
        labels = dc_data.y
        
        for i in range(len(ids)):
            smi_canonical = str(ids[i])
            label = int(labels[i])
            label_str = "Yes" if label == 1 else "No"
            
            # List of SMILES to train on
            smiles_list = [smi_canonical]
            
            if augment:
                # Add 3 random variations per molecule
                for _ in range(3):
                    smiles_list.append(get_random_smiles(smi_canonical))
            
            # Tokenize all versions
            for smi in smiles_list:
                txt = f"Task: BBBP Permeability | SMILES: {smi} | Permeable: {label_str}" + tokenizer.eos_token
                
                enc = tokenizer(
                    txt, 
                    max_length=max_len, 
                    padding="max_length", 
                    truncation=True, 
                    return_tensors="pt"
                )
                self.data.append({
                    "ids": enc["input_ids"][0],
                    "mask": enc["attention_mask"][0],
                    "label": label
                })
                
        # Shuffle data to mix augmented versions
        if augment:
            np.random.shuffle(self.data)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# 4. PREPARE NEW LOADERS
print("[*] Generating Augmented Training Data (Randomized SMILES)...")
# Train par Augment=True, Test par False (Strict Evaluation)
train_ds_aug = AugmentedBBBPDataset(train_dc, tokenizer, augment=True)
test_ds_clean = AugmentedBBBPDataset(test_dc, tokenizer, augment=False)

train_loader_aug = DataLoader(train_ds_aug, batch_size=2, shuffle=True)
print(f"Original Samples: {len(train_dc)} -> Augmented Training Samples: {len(train_ds_aug)}")

# 5. RETRAINING LOOP (3 Epochs Only - Sweet Spot)
EPOCHS = 3
ACCUM_STEPS = 8 

print(f"[*] Starting Augmented Training...")

best_auc = 0.0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader_aug, desc=f"Ep {epoch+1}")
    
    for step, batch in enumerate(pbar):
        ids = batch["ids"].to("cuda")
        mask = batch["mask"].to("cuda")
        
        out = model(input_ids=ids, attention_mask=mask, labels=ids)
        loss = out.loss / ACCUM_STEPS
        loss.backward()
        
        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            optimizer.zero_grad()
            
        epoch_loss += loss.item() * ACCUM_STEPS
        pbar.set_postfix({"loss": f"{epoch_loss/(step+1):.4f}"})

    # Validation
    print("Validating...")
    model.eval()
    preds, acts = [], []
    id_yes = tokenizer.encode("Yes", add_special_tokens=False)[0]
    id_no = tokenizer.encode("No", add_special_tokens=False)[0]
    
    with torch.no_grad():
        for item in tqdm(test_ds_clean.data, desc="Testing"):
            full_txt = tokenizer.decode(item["ids"], skip_special_tokens=True)
            query = full_txt.split("Permeable:")[0] + "Permeable:"
            
            inp = tokenizer(query, return_tensors="pt").to("cuda")
            out = model(**inp)
            logits = out.logits[0, -1, [id_no, id_yes]]
            probs = torch.nn.functional.softmax(logits.float(), dim=-1)
            
            preds.append(probs[1].item())
            acts.append(item["label"])
            
    auc = roc_auc_score(acts, preds)
    print(f"Epoch {epoch+1} BBBP ROC-AUC: {auc:.4f}")
    
    # Save ANY improvement over 0.70
    if auc > 0.70:
        print(f"SOTA BROKEN! Saving Model ({auc:.4f})...")
        model.save_pretrained(f"mistral_bbbp_aug_epoch_{epoch+1}")
        best_auc = auc

print(f"Final Augmented Score: {best_auc:.4f}")

[*] Generating Augmented Training Data (Randomized SMILES)...


[16:58:53] WARNING: not removing hydrogen atom without neighbors
[16:58:53] WARNING: not removing hydrogen atom without neighbors
[16:58:53] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not removing hydrogen atom without neighbors
[16:58:54] WARNING: not r

Original Samples: 1631 -> Augmented Training Samples: 6524
[*] Starting Augmented Training...


Ep 1:   0%|          | 0/3262 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Ep 1: 100%|██████████| 3262/3262 [1:42:24<00:00,  1.88s/it, loss=6.0380]


Validating...


...

Epoch 2 BBBP ROC-AUC: 0.6687


Ep 3:   0%|          | 0/3262 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Ep 3: 100%|██████████| 3262/3262 [1:42:24<00:00,  1.88s/it, loss=6.0020]


Validating...


Testing: 100%|██████████| 204/204 [00:44<00:00,  4.60it/s]


Epoch 3 BBBP ROC-AUC: 0.7141
SOTA BROKEN! Saving Model (0.7141)...
Final Augmented Score: 0.7141
